<a href="https://colab.research.google.com/github/andluizsouza/unicamp-llm-agents/blob/main/modules/04_projeto_pratico/step_01/E1_souza.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# Entregável 1 — Especificação e Baseline

> **Grupo:** Anderson Luiz Brandão de Souza
> **Tema/Projeto:** RecFair — recomendação de vitrine com contrato utilidade + justiça
> **Data:** 07/09/2026

Este notebook contém a **especificação inicial**, um **baseline stuffing** (uma chamada a um LLM com catálogo e vendas no prompt), um **golden-set congelado** e a análise crítica das limitações. As siglas (E1, T01, `e1_slice`, G-, …) estão no **glossário** abaixo.

**Antes de enviar:** `Runtime → Run all` e salvar com saídas visíveis. Nenhuma chave de API neste arquivo.

**Arquivos ao lado deste notebook (envie juntos no Colab):** `generate_catalog.py` e, se já gerados, `data/tb_catalogo.csv` e `data/tb_vendas.csv`.


# Glossário

Siglas e identificadores usados neste notebook e nas entregas seguintes.

| Termo | Significado |
| :--- | :--- |
| **E1, E2, E3, E4** | Entregáveis da disciplina (1 = spec + baseline; 2 = workflow/tools; 3 = multiagente; 4 = comparação final). |
| **V1** | Versão 1 do sistema = o baseline deste E1 (stuffing, uma chamada, sem tools). |
| **UC** | Caso de uso (UC1 consulta por categoria, UC2 categoria+marca, …). |
| **RF / RNF** | Requisito funcional / não funcional. **RF-S** = requisito do *sistema* (visão), medido já no E1 mas com sucesso não exigido nesta versão. |
| **T01 … T14** | Itens do **golden-set** (conjunto de testes congelado). T01 é o primeiro caso; os IDs não mudam entre E1–E4. |
| **e1_slice** | Taxa de aprovação só em T01–T10 (família S). É a régua de promoção do E1. T11–T14 ficam de fora dessa média. |
| **S_** | Família de caso que o V1 deve acertar: `S_exact` (ordem 7d), `S_soft` (filtro + SKUs válidos), `S_diversity` (filtro + **≥2 marcas**), `S_window` (não usar o mês inteiro), `S_abstain` (abster com o `reason` certo). |
| **G-** | Caso de **gap** do sistema (falha previsível no E1): `G-claim` / `G_need` (benefício sem ficha), `G-preço` / `G_price` (orçamento sem preço praticado). |
| **SKU** | Código opaco de 6 caracteres no catálogo (`24A51X`, `6D2W9K`, …). Não é sequencial nem mnemônico. |
| **Top-5 / N** | Cardinalidade da vitrine (`N=5`). |
| **units_7d** | Soma de `qt_sold` na janela 25–31/08/2026 (sem TODAY). |
| **TODAY** | Data congelada da consulta (`2026-09-01`). |
| **base_price** | Preço *base de tabela* no CSV/SQLite. Não é o preço praticado (este virá de tool/API). |
| **golden-set / golden_revision** | Conjunto de casos congelado; hash SHA-256 curto do JSON dos casos. |
| **halt_reason** | Por que a execução parou: `completed`, `abstained`, `schema_invalid`. |
| **stuffing** | Colocar as tabelas inteiras no prompt (sem SQL/RAG). |
| **HITL** | Humano no loop (publica a vitrine; o agente não compra). |
| **RAG / MCP / PII** | Recuperação aumentada por documentos; protocolo de tools remotas; dados pessoais. |


# 1. Descrição do problema

Sistemas de recomendação em e-commerce tendem a maximizar clique e conversão. Na prática isso reforça **viés de popularidade** (poucos SKUs e marcas dominam a vitrine) e **estereótipo** quando o perfil do cliente é inferido. Claims (fixação, vegano, anticaspa, ocasião noturna) saem sem evidência.

O **RecFair** trata o problema no ponto de decisão: recomendar do catálogo da loja, explicar com fonte, auditar concentração de exposição e recusar claim sem evidência. A conversão é sinal, não objetivo único. O agente **não executa compra**; um humano publica a vitrine.

**Nesta primeira versão (V1)** o recorte é deliberadamente estreito: dada uma necessidade em português, devolver o **Top-5 da categoria** (e da marca, se pedida) segundo as **unidades vendidas nos últimos 7 dias completos**, com abstenção quando a categoria não for determinável. Preço vigente, estoque, fichas de benefício/claim, fairness rígida e multi-turno ficam para incrementos medidos.


# 2. Usuário-alvo e stakeholders

| Papel | Objetivo |
| :--- | :--- |
| **Usuário principal:** consumidor no chat | Descrever a necessidade em pt-BR e receber uma lista útil e honesta. |
| **Secundário:** analista de merchandising | Publicar a vitrine (humano no loop). |
| **Stakeholder:** compliance | Sem PII real; sem completar perfil identitário; claims só com evidência (visão). |

O agente não é caixa, SAC de pedido/frete nem classificador demográfico.


# 3. Casos de uso principais

Ator = consumidor. Só UC1–UC2 são sucesso **esperado** do baseline.

| ID | Entrada | Objetivo | Saída de sucesso | E1 |
| :--- | :--- | :--- | :--- | :--- |
| UC1 | Categoria ampla (“produtos de cabelo mais vendidos”) | Top-5 da categoria na janela de 7 dias | 5 SKUs do catálogo na categoria e **≥2 marcas** (há alternativas com venda na janela) | sim |
| UC2 | Categoria + marca (“cabelos da Match”) | Top-5 filtrado | 5 SKUs da marca na categoria, ordem da janela | sim |
| UC3 | Teto de preço | Respeitar orçamento | Só itens com preço vigente ≤ teto | não (tool) |
| UC4 | Benefício / claim (“anticaspa”, “vegano”, “alta fixação”) | Adequar à ficha | Itens cuja ficha respalda o claim | não (RAG) |
| UC5 | Pedido sem categoria / “perfume” sem gênero / marca multi-categoria | Não chutar | `missing_category` | sim |
| UC6 | Fora do sortimento (“protetor solar”) | Recusar | `unknown_category` | sim |


# 4. Escopo, não-objetivos e premissas

| | |
| :--- | :--- |
| **Escopo V1** | Uma consulta por execução; catálogo sintético de 40 SKUs (inspirado em produtos públicos de [O Boticário](https://www.boticario.com.br/), SKUs inventados); stuffing de `tb_catalogo` (sem preço) + `tb_vendas` diárias de agosto/2026; Top-5 ou abstenção. |
| **Não-objetivos** | RAG, SQL/tools no agente, MCP, LangGraph, memória de sessão, compra, PII real, UI web, preço/estoque no agente, documentos de claim no contexto. |
| **Premissas** | Catálogo + vendas cabem na janela do modelo; `TODAY = 2026-09-01`; janela de ranking = **2026-08-25 a 2026-08-31** (7 dias completos, **sem** TODAY); `base_price` no CSV/SQLite é o **preço base de tabela**, não o preço praticado (este virá de tool/API no E2+). Geração dos dados **sem aleatoriedade**: CSV e SQLite saem do mesmo `build_dataset()`. |

Fichas (PDF/docs) de benefício e claim **existem só na visão**: o avaliador do E1 não as injeta no prompt. Casos G no golden-set documentam a falha previsível.


# 5. Entradas e saídas

## Entrada

Uma string em pt-BR (necessidade do cliente). Sem histórico de sessão.

## Saída (contrato largo, estável até o E4)

Campos futuros nascem `null` no E1 para não renegociar o schema.

```text
status: recommendation | abstention
items: 0 ou exatamente 5 × {
  sku, name, brand, category, units_7d,   # V1 preenche
  price_brl, in_stock, is_launch, is_promo,  # sempre null no E1
  explanation, citations, fairness_notes     # sempre null no E1
}
reason: missing_category | unknown_category | unknown_brand | null
halt_reason: completed | abstained | schema_invalid
```

`units_7d` é a soma de `qt_sold` na janela 25–31/08. **`in_stock` fica null**: o E1 não tem estoque.


# 6. Requisitos funcionais

Verificáveis. Fatia V1 (o baseline deve passar) vs. sistema (medidos, sucesso não exigido).

| ID | Requisito | Critério (E1) | Verificação |
| :--- | :--- | :--- | :--- |
| **RF-01** | Só SKUs do catálogo | 0 SKU inventado nos casos com lista | determinística |
| **RF-02** | Filtro de categoria (e marca se pedida) | ≥80% dos casos S com lista: os 5 SKUs respeitam o filtro | determinística |
| **RF-03** | Ranking 7d + desempate `cod_sku` | Nos casos com marca especificada (T03): ordem idêntica ao gabarito pandas | determinística |
| **RF-04** | Abster sem categoria segura | T06, T08, T09 com `reason=missing_category` | determinística |
| **RF-05** | Categoria ou marca inexistente | T07 `unknown_brand`; T10 `unknown_category` | determinística |
| **RF-06** | Não usar o mês inteiro | T05: o campeão do mês (Clash, `6D2W9K`) **não** entra no Top-5 | determinística |
| **RF-07** | Diversidade quando a marca **não** foi pedida | Se a categoria tem ≥2 marcas com venda na janela, o Top-5 deve ter **n_brands ≥ 2**. Cinco itens da mesma marca é **erro** (T01, T02, T04, T05). | determinística |
| **RF-S01** | Teto de preço | T12 — esperado **falhar** (sem preço no prompt) | observação |
| **RF-S02** | Claim / benefício | T11, T13, T14 — esperado **falhar** (sem ficha) | observação |

O gabarito *de popularidade* de T01/T02 é 5× Match, de T04 é 5× Cuide-se Bem e de T05 é 5× Malbec: isso documenta o viés, **não** é a saída aceita. O V1 deve substituir pelo menos um item por outra marca com `units_7d > 0` (em cabelos, p.ex. Cachos de Uva). Quando o cliente **pede** a marca (T03), 5× Match é correto.


# 7. Requisitos não funcionais e restrições

| ID | Requisito | Régua |
| :--- | :--- | :--- |
| **RNF-01** | Saída no schema Pydantic | 100% dos casos (senão `schema_invalid`) |
| **RNF-02** | Sem chaves/PII no notebook | inspeção |
| **RNF-03** | Latência mediana do agente | &lt; 10 s |
| **RNF-04** | Caminho feliz | 1 chamada LLM; 0 tools |
| **RNF-05** | Reproducibilidade | `temperature=0`, `prompt_version=v1`, `TODAY` e CSVs congelados, `golden_revision` |
| **RNF-06** | HITL | o agente não compra nem publica vitrine |
| **RNF-07** | Sem memória | cada `baseline(query)` é independente |


# 8. Recursos externos potencialmente necessários

| Recurso | Nesta entrega | Hipótese futura |
| :--- | :--- | :--- |
| Google Gemini (`model_version`, default `gemini-3.5-flash`) | sim | mesmo modelo nas comparações |
| CSVs `tb_catalogo` / `tb_vendas` | sim (gerados por `generate_catalog.py`) | mesmas tabelas no SQLite (E2, text-to-SQL) |
| `base_price` no catálogo | arquivo/SQLite apenas; **não** vai ao prompt | preço base de tabela; praticado = tool/API |
| Adapters preço / estoque | não no agente | E2 |
| Fichas PDF/docs por SKU | não | RAG (claims/benefícios) |
| MCP | não | se preço/estoque forem reusados por ≥2 nós |
| LangGraph | não | E2 (disciplina) |


# 9. Tipo de baseline escolhido

**Parcial.**

1. **Adequação.** A tarefa nuclear é “vitrine a partir do catálogo”. Popularidade nos últimos 7 dias é o recorte clássico de varejo **e** o viés que o RecFair quer combater. Uma chamada com stuffing é a solução mais simples que ainda lê fatos tabulares (não inventa o sortimento).
2. **Simplificado de propósito.** Sem SQL, sem preço/estoque, sem ficha, sem grafo, sem sessão. O prompt é honesto: somar a janela 25–31/08, filtrar categoria/marca, abster se a categoria for insegura, e **não** devolver cinco itens da mesma marca quando houver alternativas na janela.
3. **Comparação futura.** Mesmo golden-set (só cresce), mesmo schema, mesmas funções de verify. O stuffing de ~1 240 linhas diárias justifica text-to-SQL no E2: o modelo tem de filtrar a janela e somar — erro de agregação é limitação medida, não espantalho.


# 10. Critérios preliminares de sucesso

| Critério | RF/RNF | Como se mede |
| :--- | :--- | :--- |
| SKU válido | RF-01 | todo `sku` ∈ catálogo |
| Filtro | RF-02 | categoria (e marca) batem |
| Ordem 7d | RF-03 | igualdade com pandas nos casos *exact_rank* |
| Abstenção | RF-04, RF-05 | `status` + `reason` |
| Janela 7d ≠ mês | RF-06 | Clash (`6D2W9K`) ausente em T05 |
| Diversidade | RF-07 | `n_brands ≥ 2` se `require_diversity` (marca não pedida e há alternativas) |
| Preço / claim | RF-S01, RF-S02 | casos G: falha **esperada** |
| Latência | RNF-03 | mediana s |
| Custo / tokens | RNF-04 | usage_metadata × preço indicativo |
| Schema | RNF-01 | parse Pydantic |

Duas leituras do mesmo run: **`e1_slice`** = taxa de aprovação só em T01–T10 (família S). Dimensões G (T11–T14) **não** derrubam a fatia V1.


# 11. Configuração do ambiente


In [ ]:
%pip install -q -U langchain langchain-google-genai pydantic pandas


In [ ]:
import datetime
import getpass
import hashlib
import json
import os
import platform
import sys
import time
from pathlib import Path

import pandas as pd
from pydantic import BaseModel, Field, field_validator
from typing import Literal


def _apply_dotenv() -> None:
    """Load KEY=VALUE from nearby .env files without overriding existing env."""
    seen: set[Path] = set()
    for root in [Path.cwd(), *Path.cwd().parents[:5]]:
        path = root / ".env"
        if path in seen or not path.is_file():
            continue
        seen.add(path)
        for raw in path.read_text(encoding="utf-8").splitlines():
            line = raw.strip()
            if not line or line.startswith("#") or "=" not in line:
                continue
            key, _, val = line.partition("=")
            key, val = key.strip(), val.strip().strip('"').strip("'")
            if key and key not in os.environ:
                os.environ[key] = val


def carregar_chave_google() -> str:
    """Resolve GOOGLE_API_KEY via env, Colab userdata ou prompt. Nunca imprime a chave."""
    if os.environ.get("GOOGLE_API_KEY"):
        return "variável de ambiente GOOGLE_API_KEY"
    if os.environ.get("GEMINI_API_KEY"):
        os.environ["GOOGLE_API_KEY"] = os.environ["GEMINI_API_KEY"]
        return "variável de ambiente GEMINI_API_KEY"
    try:
        from google.colab import userdata  # type: ignore

        for secret_name in ("GOOGLE_API_KEY", "GEMINI_API_KEY"):
            try:
                value = userdata.get(secret_name)
            except Exception:
                continue
            if value:
                os.environ["GOOGLE_API_KEY"] = value
                return f"Colab userdata:{secret_name}"
    except ImportError:
        pass
    os.environ["GOOGLE_API_KEY"] = getpass.getpass("GOOGLE_API_KEY: ")
    return "entrada manual"


_apply_dotenv()
origem_chave = carregar_chave_google()
assert os.environ.get("GOOGLE_API_KEY"), "Chave Google/Gemini não configurada."
print("Chave carregada via:", origem_chave)


In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI

# Altere aqui (ou exporte RECFAIR_MODEL_VERSION) sem editar o restante do notebook.
model_version = os.environ.get("RECFAIR_MODEL_VERSION", "gemini-3.5-flash")
TEMPERATURE = 0
PROMPT_VERSAO = "v1"
N_RECOMMEND = 5

# Gemini 3.x pode defaultar temperature=1; forçamos 0 explicitamente.
llm = ChatGoogleGenerativeAI(model=model_version, temperature=TEMPERATURE)

RUN_INFO = {
    "architecture_id": "baseline",
    "architecture_date": "2026-09-07",
    "modelo": model_version,
    "model_version": model_version,
    "temperatura": TEMPERATURE,
    "prompt_versao": PROMPT_VERSAO,
    "data": datetime.datetime.now().isoformat(timespec="seconds"),
    "python": platform.python_version(),
    "n": N_RECOMMEND,
    "today": "2026-09-01",
    "window": ["2026-08-25", "2026-08-31"],
}
RUN_INFO


# 12. Dados ou entradas de exemplo

Catálogo sintético de **40 SKUs** (10 perfumaria masculina, 10 feminina, 10 corpo e banho, 10 cabelos), inspirado no sortimento público de O Boticário. Cada `cod_sku` é uma string **opaca de 6 caracteres** (ex. `24A51X`): fixa, não sequencial, não mnemônica, idêntica no CSV e no SQLite.

Para RF-07 ser testável, o Top-5 de popularidade 7d está plantado com **5 SKUs da mesma marca** em cabelos (Match), corpo e banho (Cuide-se Bem) e perfumaria masculina (Malbec). Há marcas alternativas com venda na janela em cada uma dessas categorias.

- `tb_catalogo.csv`: `cod_sku, name_sku, brand, category, base_price`
- `tb_vendas.csv`: `date, cod_sku, qt_sold` — 01/08/2026 a 31/08/2026 (1 240 linhas)
- **`base_price` não entra no prompt.** É o preço base de tabela; o preço praticado será uma tool/API depois.
- Público-alvo e claims **não** estão nestes CSVs; virão em documentos para RAG.
- `generate_catalog.py` não usa RNG: `build_dataset()` alimenta CSV e SQLite com as **mesmas** linhas (`write_sqlite()` confere a igualdade).


In [ ]:
STEP_CANDIDATES = [
    Path.cwd(),
    Path.cwd() / "modules" / "04_projeto_pratico" / "step_01",
    Path("/content/modules/04_projeto_pratico/step_01"),
]
STEP_DIR = next((p for p in STEP_CANDIDATES if (p / "generate_catalog.py").exists()), Path.cwd())
if str(STEP_DIR) not in sys.path:
    sys.path.insert(0, str(STEP_DIR))

from generate_catalog import (  # noqa: E402
    CATEGORY_BODY,
    CATEGORY_HAIR,
    CATEGORY_PERFUME_F,
    CATEGORY_PERFUME_M,
    SKU_ANTICASPA,
    SKU_CLASH,
    TODAY,
    WINDOW_END,
    WINDOW_START,
    catalog_by_sku,
    ensure_csv_files,
    generate_sales,
    gold_top_n,
    units_in_window,
)

DATA_DIR = STEP_DIR / "data"
paths = ensure_csv_files(DATA_DIR)
print("STEP_DIR:", STEP_DIR)
print("CSVs:", {k: str(v) for k, v in paths.items()})

df_catalogo = pd.read_csv(paths["tb_catalogo"])
df_vendas = pd.read_csv(paths["tb_vendas"])
assert len(df_catalogo) == 40
assert "base_price" in df_catalogo.columns
assert "list_price" not in df_catalogo.columns
assert df_vendas["date"].min() == "2026-08-01"
assert df_vendas["date"].max() == "2026-08-31"
print("catálogo:", df_catalogo.shape, "vendas:", df_vendas.shape)
df_catalogo.head()


In [ ]:
sales_rows = generate_sales()
print(f"TODAY={TODAY} | janela ranking={WINDOW_START} .. {WINDOW_END} (TODAY excluído)")
print("\nGabarito de *popularidade* (units_7d DESC, sku ASC). Em cabelos isso dá 5× Match — copiar essa lista em T01 é ERRO de diversidade:\n")
for cat in (CATEGORY_PERFUME_M, CATEGORY_PERFUME_F, CATEGORY_BODY, CATEGORY_HAIR):
    print(f"=== {cat} ===")
    for i, row in enumerate(gold_top_n(sales_rows, category=cat), start=1):
        print(f"  {i}. {row['cod_sku']}  {row['units_7d']:4d}  {row['brand']:16s}  {row['name_sku']}")
    print()

tb_catalogo_prompt = df_catalogo[["cod_sku", "name_sku", "brand", "category"]].to_csv(index=False)
tb_vendas_prompt = df_vendas[["date", "cod_sku", "qt_sold"]].to_csv(index=False)
print("chars no prompt: catálogo", len(tb_catalogo_prompt), "| vendas", len(tb_vendas_prompt))


# 13. Implementação do baseline

Stuffing: uma chamada, prompt versionado `v1`, tabelas interpoladas com `.format`. Sem tools. Sem memória.


In [ ]:
class RecommendationItem(BaseModel):
    sku: str
    name: str
    brand: str
    category: str
    units_7d: int
    price_brl: float | None = None
    in_stock: bool | None = None
    is_launch: bool | None = None
    is_promo: bool | None = None
    explanation: str | None = None
    citations: list[str] | None = None
    fairness_notes: str | None = None


class RecFairOutput(BaseModel):
    status: Literal["recommendation", "abstention"]
    items: list[RecommendationItem] = Field(default_factory=list)
    reason: Literal["missing_category", "unknown_category", "unknown_brand"] | None = None
    halt_reason: Literal["completed", "abstained", "schema_invalid"]

    @field_validator("items")
    @classmethod
    def _five_or_empty(cls, items: list[RecommendationItem]) -> list[RecommendationItem]:
        if items and len(items) != N_RECOMMEND:
            raise ValueError(f"recommendation must have exactly {N_RECOMMEND} items")
        return items


PROMPT_TEMPLATE = """Você é o RecFair, assistente de vitrine de e-commerce.

Data da consulta (TODAY): {today}
Janela de ranking: últimos 7 dias COMPLETOS, SEM incluir TODAY: {window_start} a {window_end} (inclusive).
Some apenas qt_sold nessa janela. Não use o mês inteiro nem vendas de TODAY.

Catálogo (tb_catalogo):
{tb_catalogo}

Vendas diárias (tb_vendas):
{tb_vendas}

Tarefa: para a consulta do cliente, devolva exatamente 5 SKUs (Top-5) OU abstenha.

Regras:
1. Interprete a categoria como no máximo um de: perfumaria_masculina, perfumaria_feminina, corpo_e_banho, cabelos.
2. Se a categoria não puder ser determinada com segurança (ausente; "perfume(s)" sem gênero; marca que aparece em várias categorias sem categoria explícita), status=abstention e reason=missing_category.
3. Categoria pedida que não existe no catálogo: reason=unknown_category.
4. Marca pedida que não existe no catálogo: reason=unknown_brand.
5. Filtre pela categoria (obrigatória) e pela marca (somente se o cliente a especificou).
6. Ordene por soma de qt_sold na janela (maior primeiro). Empate: cod_sku ascendente.
7. Se o cliente NÃO especificou marca e existirem SKUs de outras marcas com vendas na janela na mesma categoria, é ERRO devolver os 5 itens da mesma marca. Inclua pelo menos duas marcas. A utilidade (vendas na janela) continua o critério de ordenação entre os candidatos.
8. Não invente SKU, nome, marca ou units_7d. units_7d deve ser a soma da janela.
9. Deixe price_brl, in_stock, is_launch, is_promo, explanation, citations e fairness_notes como null. Você não tem preço praticado, estoque, benefícios nem claims.
10. Não complete o perfil do cliente com estereótipo. Sem memória de turnos anteriores.

Consulta do cliente:
{query}
"""


def build_prompt(query: str) -> str:
    return PROMPT_TEMPLATE.format(
        today=TODAY.isoformat(),
        window_start=WINDOW_START.isoformat(),
        window_end=WINDOW_END.isoformat(),
        tb_catalogo=tb_catalogo_prompt,
        tb_vendas=tb_vendas_prompt,
        query=query.strip(),
    )


structured_llm = llm.with_structured_output(RecFairOutput, include_raw=True)


def _usage(raw) -> tuple[int | None, int | None]:
    meta = getattr(raw, "usage_metadata", None) or {}
    if not isinstance(meta, dict):
        return (
            getattr(meta, "input_tokens", None),
            getattr(meta, "output_tokens", None),
        )
    return meta.get("input_tokens"), meta.get("output_tokens")


def baseline(entrada: str) -> tuple[RecFairOutput, dict]:
    """Uma chamada Gemini: stuffing das tabelas + consulta. Sem tools e sem sessão."""
    prompt = build_prompt(entrada)
    inicio = time.perf_counter()
    try:
        packed = structured_llm.invoke(prompt)
        latencia = time.perf_counter() - inicio
        if isinstance(packed, dict) and "parsed" in packed:
            raw = packed.get("raw")
            parsed = packed.get("parsed")
            tokens_in, tokens_out = _usage(raw)
        else:
            parsed = packed
            tokens_in, tokens_out = None, None
        if parsed is None:
            parsed = RecFairOutput(status="abstention", reason=None, halt_reason="schema_invalid")
        metricas = {
            "latencia_s": round(latencia, 2),
            "tokens_entrada": tokens_in,
            "tokens_saida": tokens_out,
            "chamadas_llm": 1,
            "tool_calls": 0,
        }
        return parsed, metricas
    except Exception as exc:
        latencia = time.perf_counter() - inicio
        fallback = RecFairOutput(
            status="abstention",
            reason=None,
            halt_reason="schema_invalid",
        )
        metricas = {
            "latencia_s": round(latencia, 2),
            "tokens_entrada": None,
            "tokens_saida": None,
            "chamadas_llm": 1,
            "tool_calls": 0,
            "erro": type(exc).__name__,
        }
        return fallback, metricas


print("prompt_version:", PROMPT_VERSAO, "| chars template:", len(PROMPT_TEMPLATE))


# 14. Conjunto de avaliação

Escrito **antes** de tunar o prompt. Congelado: os mesmos IDs serão reexecutados nos E2–E4. Novos casos só por acréscimo.

| ID | Tipo | Entrada (resumo) | Esperado V1 |
| :--- | :--- | :--- | :--- |
| T01 | normal + **diversidade** | produtos de cabelo mais vendidos | 5 SKUs em `cabelos` e **n_brands ≥ 2** (5× Match = reprovação) |
| T02 | paráfrase | o que mais sai de cabelo nesta semana | idem T01 (diversidade) |
| T03 | composto / marca | cabelos da linha Match | ordem exata do gabarito Match (5× Match **ok**) |
| T04 | normal + diversidade | corpo e banho mais vendidos | 5 SKUs na categoria e **n_brands ≥ 2** |
| T05 | janela 7d + diversidade | colônias masculinas mais vendidas | sem Clash (`6D2W9K`); **n_brands ≥ 2** |
| T06 | informação ausente | “algo bom” / presente | `missing_category` |
| T07 | marca inexistente | shampoos da Zorblax | `unknown_brand` |
| T08 | ambíguo | perfumes mais vendidos | `missing_category` |
| T09 | ambíguo (marca multi-cat.) | mais vendidos da Malbec | `missing_category` |
| T10 | fora de sortimento | protetor solar até 50 | `unknown_category` |
| T11 | G-claim | shampoo anticaspa | falha previsível (sem ficha) |
| T12 | G-preço | colônia masculina até R$180 | falha previsível (sem preço) |
| T13 | G-claim | hidratante vegano | falha previsível |
| T14 | G-claim | perfume feminino de alta fixação / noite | falha previsível |

T03 exige ordem (`S_exact`) porque a marca está **pedida**. T01/T02 são `S_diversity`: o ouro de popularidade (5× Match) existe para mostrar o viés; **não** é a resposta certa.

## Chaves de cada item em `test_cases`

| Chave | Obrigatória | Significado |
| :--- | :--- | :--- |
| `id` | sim | Identificador estável (`T01`…). Não reutilizar nem apagar. |
| `tipo` | sim | Rótulo humano (normal, paráfrase, ambíguo, G-claim, …). |
| `familia` | sim | Qual ramo do `verify_case` se aplica (`S_exact`, `S_soft`, `S_diversity`, `S_window`, `S_abstain`, `G_need`, `G_price`). |
| `verificacao` | sim | Sempre `auto` neste E1 (sem rubrica humana). |
| `entrada` | sim | Query em pt-BR enviada ao baseline. |
| `e1_slice` | sim | `True` = entra na média de promoção do E1 (T01–T10). `False` = gap G. |
| `category` | se houver lista | Categoria canônica esperada no filtro (`perfumaria_masculina`, …). |
| `brand` | se o cliente pediu marca | Marca a filtrar; `None` = não pediu (aí vale diversidade). |
| `require_diversity` | se `brand` é `None` e há lista | `True` = reprova vitrine com uma só marca quando há alternativas na janela. |
| `expected_reason` | em `S_abstain` | Código de abstenção (`missing_category`, `unknown_brand`, `unknown_category`). |
| `forbidden_skus` | em `S_window` | SKUs que não podem aparecer (`6D2W9K` Clash = campeão do mês, não da janela). |
| `target_sku` | em G-claim | SKU que *deveria* ser escolhido com ficha (ex. `H8Q3N1` anticaspa); só observação no E1. |


In [ ]:
CATALOG = catalog_by_sku()

test_cases = [
    {
        "id": "T01",
        "tipo": "normal",
        "familia": "S_diversity",
        "verificacao": "auto",
        "entrada": "Quais os produtos de cabelo mais vendidos?",
        "category": CATEGORY_HAIR,
        "brand": None,
        "require_diversity": True,
        "e1_slice": True,
    },
    {
        "id": "T02",
        "tipo": "paráfrase",
        "familia": "S_diversity",
        "verificacao": "auto",
        "entrada": "Me indica o que mais sai de cabelo nesta semana.",
        "category": CATEGORY_HAIR,
        "brand": None,
        "require_diversity": True,
        "e1_slice": True,
    },
    {
        "id": "T03",
        "tipo": "composto",
        "familia": "S_exact",
        "verificacao": "auto",
        "entrada": "Quais os produtos de cabelo da Match mais vendidos?",
        "category": CATEGORY_HAIR,
        "brand": "Match",
        "e1_slice": True,
    },
    {
        "id": "T04",
        "tipo": "normal",
        "familia": "S_diversity",
        "verificacao": "auto",
        "entrada": "Quais os produtos de corpo e banho mais vendidos?",
        "category": CATEGORY_BODY,
        "brand": None,
        "require_diversity": True,
        "e1_slice": True,
    },
    {
        "id": "T05",
        "tipo": "janela_7d",
        "familia": "S_window",
        "verificacao": "auto",
        "entrada": "Quais as colônias masculinas mais vendidas?",
        "category": CATEGORY_PERFUME_M,
        "brand": None,
        "forbidden_skus": [SKU_CLASH],
        "require_diversity": True,
        "e1_slice": True,
    },
    {
        "id": "T06",
        "tipo": "informação ausente",
        "familia": "S_abstain",
        "verificacao": "auto",
        "entrada": "Me recomenda algo bom para presentear.",
        "expected_reason": "missing_category",
        "e1_slice": True,
    },
    {
        "id": "T07",
        "tipo": "informação ausente",
        "familia": "S_abstain",
        "verificacao": "auto",
        "entrada": "Quero os shampoos mais vendidos da Zorblax.",
        "expected_reason": "unknown_brand",
        "e1_slice": True,
    },
    {
        "id": "T08",
        "tipo": "ambíguo",
        "familia": "S_abstain",
        "verificacao": "auto",
        "entrada": "Quais os perfumes mais vendidos?",
        "expected_reason": "missing_category",
        "e1_slice": True,
    },
    {
        "id": "T09",
        "tipo": "ambíguo",
        "familia": "S_abstain",
        "verificacao": "auto",
        "entrada": "Quais os mais vendidos da Malbec?",
        "expected_reason": "missing_category",
        "e1_slice": True,
    },
    {
        "id": "T10",
        "tipo": "fora de escopo",
        "familia": "S_abstain",
        "verificacao": "auto",
        "entrada": "Preciso de um protetor solar até 50 reais.",
        "expected_reason": "unknown_category",
        "e1_slice": True,
    },
    {
        "id": "T11",
        "tipo": "G-claim",
        "familia": "G_need",
        "verificacao": "auto",
        "entrada": "Quero um shampoo anticaspa.",
        "category": CATEGORY_HAIR,
        "brand": None,
        "target_sku": SKU_ANTICASPA,
        "e1_slice": False,
    },
    {
        "id": "T12",
        "tipo": "G-preço",
        "familia": "G_price",
        "verificacao": "auto",
        "entrada": "Colônia masculina até 180 reais, as mais vendidas.",
        "category": CATEGORY_PERFUME_M,
        "brand": None,
        "e1_slice": False,
    },
    {
        "id": "T13",
        "tipo": "G-claim",
        "familia": "G_need",
        "verificacao": "auto",
        "entrada": "Me indica um hidratante corporal vegano.",
        "category": CATEGORY_BODY,
        "brand": None,
        "e1_slice": False,
    },
    {
        "id": "T14",
        "tipo": "G-claim",
        "familia": "G_need",
        "verificacao": "auto",
        "entrada": "Perfume feminino de alta fixação para ocasião noturna.",
        "category": CATEGORY_PERFUME_F,
        "brand": None,
        "e1_slice": False,
    },
]

blob = json.dumps(test_cases, sort_keys=True, ensure_ascii=False).encode("utf-8")
golden_revision = hashlib.sha256(blob).hexdigest()[:16]
RUN_INFO["golden_revision"] = golden_revision
print(len(test_cases), "casos | golden_revision =", golden_revision)


# 15. Implementação da verificação


In [ ]:
def gold_for(caso: dict) -> list[str]:
    if caso.get("category") is None:
        return []
    rows = gold_top_n(
        sales_rows,
        category=caso["category"],
        brand=caso.get("brand"),
        n=N_RECOMMEND,
    )
    return [r["cod_sku"] for r in rows]


def verify_case(caso: dict, saida: RecFairOutput) -> dict:
    """Deterministic checks. G cases never block e1_slice."""
    gold = gold_for(caso)
    skus = [item.sku for item in saida.items]
    invented = [s for s in skus if s not in CATALOG]
    cats_ok = True
    brands_ok = True
    if saida.status == "recommendation" and caso.get("category"):
        cats_ok = all(CATALOG[s]["category"] == caso["category"] for s in skus if s in CATALOG)
        if caso.get("brand"):
            brands_ok = all(CATALOG[s]["brand"] == caso["brand"] for s in skus if s in CATALOG)
    n_brands = len({CATALOG[s]["brand"] for s in skus if s in CATALOG}) if skus else 0
    gold_brands = len({CATALOG[s]["brand"] for s in gold}) if gold else 0
    order_match = skus == gold if gold else False
    forbidden = set(caso.get("forbidden_skus") or [])
    forbidden_hit = bool(forbidden.intersection(skus))
    diversity_ok = (n_brands >= 2) if caso.get("require_diversity") else True

    familia = caso["familia"]
    lista_ok = (
        saida.status == "recommendation"
        and len(skus) == N_RECOMMEND
        and not invented
        and cats_ok
        and brands_ok
    )
    aprovado = False
    if familia == "S_exact":
        aprovado = lista_ok and order_match
    elif familia == "S_soft":
        aprovado = lista_ok
    elif familia == "S_diversity":
        aprovado = lista_ok and diversity_ok
    elif familia == "S_window":
        aprovado = lista_ok and not forbidden_hit and diversity_ok
    elif familia == "S_abstain":
        aprovado = saida.status == "abstention" and saida.reason == caso["expected_reason"]
    elif familia in {"G_need", "G_price"}:
        aprovado = False

    return {
        "aprovado": bool(aprovado),
        "gold": gold,
        "skus": skus,
        "invented": invented,
        "order_match": order_match,
        "n_brands": n_brands,
        "gold_n_brands": gold_brands,
        "diversity_ok": diversity_ok,
        "forbidden_hit": forbidden_hit,
        "g_target_hit": (caso.get("target_sku") in skus) if caso.get("target_sku") else None,
        "reason": saida.reason,
        "halt_reason": saida.halt_reason,
        "status": saida.status,
    }


print("verify pronto; exemplo T03 gold =", gold_for(test_cases[2]))


# 16. Experimentos

Cada caso: 1 chamada, latência, tokens, `halt_reason`. Sem memória entre linhas.


In [ ]:
registros = []

for caso in test_cases:
    saida, metricas = baseline(caso["entrada"])
    checagem = verify_case(caso, saida)
    registros.append(
        {
            "id": caso["id"],
            "tipo": caso["tipo"],
            "familia": caso["familia"],
            "e1_slice": caso["e1_slice"],
            "entrada": caso["entrada"],
            "saida": saida.model_dump(),
            **checagem,
            **metricas,
        }
    )
    marca = "OK" if checagem["aprovado"] else "—"
    print(f"{caso['id']}  e1={marca}  status={saida.status}  reason={saida.reason}  skus={checagem['skus']}")

print(len(registros), "execuções registradas.")


# 17. Resultados


In [ ]:
try:
    from IPython.display import display
except ImportError:
    def display(x):
        print(x)

df = pd.DataFrame(registros)
cols = [
    "id", "tipo", "familia", "e1_slice", "aprovado", "status", "reason",
    "skus", "gold", "order_match", "n_brands", "gold_n_brands", "diversity_ok",
    "forbidden_hit", "g_target_hit", "latencia_s", "tokens_entrada",
    "tokens_saida", "chamadas_llm", "halt_reason",
]
display(df[[c for c in cols if c in df.columns]])

slice_df = df[df["e1_slice"]]
g_df = df[~df["e1_slice"]]
USD_IN = 0.10 / 1_000_000
USD_OUT = 0.40 / 1_000_000
tok_in = pd.to_numeric(df["tokens_entrada"], errors="coerce").fillna(0).sum()
tok_out = pd.to_numeric(df["tokens_saida"], errors="coerce").fillna(0).sum()

RESUMO = {
    "e1_slice_aprovacao": float(slice_df["aprovado"].mean()) if len(slice_df) else None,
    "e1_slice_n": int(slice_df["aprovado"].sum()) if len(slice_df) else 0,
    "e1_slice_d": int(len(slice_df)),
    "g_target_hits": int((g_df["g_target_hit"] == True).sum()) if len(g_df) else 0,
    "latencia_mediana_s": float(df["latencia_s"].median()) if len(df) else None,
    "chamadas_llm": int(df["chamadas_llm"].sum()) if len(df) else 0,
    "tokens_entrada": int(tok_in),
    "tokens_saida": int(tok_out),
    "custo_estimado_usd": round(tok_in * USD_IN + tok_out * USD_OUT, 6),
    "nota_custo": "estimativa indicativa flash-class; conferir tabela vigente do Gemini",
}
print(json.dumps(RESUMO, indent=2, ensure_ascii=False))

out_json = STEP_DIR / "baseline_v1_resultados.json"
payload = {"run": RUN_INFO, "resumo": RESUMO, "registros": registros}
out_json.write_text(json.dumps(payload, ensure_ascii=False, indent=2, default=str), encoding="utf-8")
print("salvo:", out_json)


# 18. Análise crítica do baseline

Preencha os números após o Run all. A tese abaixo já é a esperada:

1. **O que atende.** RF-01/02/04/05 e, se a agregação da janela estiver correta, RF-06 e RF-03 em T03. Schema e 1 chamada (RNF-01, RNF-04). Sem memória (RNF-07).
2. **O que não atende.** RF-S01 preço (T12), RF-S02 claims (T11, T13, T14). Subcategoria (shampoo vs condicionador; hidratante vs sabonete) **não existe** no catálogo V1: T01 pode incluir o condicionador Match Nutrição no ouro de `cabelos`.
3. **Erros típicos do stuffing.** Somar o mês inteiro (Clash entra em T05); errar o empate Malbec Club vs Malbec Gold (ambos 210 unidades na janela); inventar `units_7d`; recusar T08 com o código errado.
4. **Entradas difíceis.** T08/T09 (ambiguidade); T01 (o ranking puro de vendas é 5× Match — o modelo precisa **abrir mão** de popularidade para cumprir RF-07).
5. **Diversidade (RF-07).** Cinco SKUs da mesma marca, havendo outras com venda na janela, é **erro** e reprova T01/T02/T04/T05. T03 (marca pedida) aceita 5× Match. Tools no E2 podem listar marcas da categoria e facilitar o cumprimento; o requisito já vale no E1.
6. **Modelo vs arquitetura.** Erro de soma/janela é do modelo sobre tabela grande. Ausência de preço, estoque e claim é da **arquitetura** (não estão no contexto).
7. **Medida.** **O baseline não consulta estoque nem preço praticado.** `in_stock` e `price_brl` são sempre `null`. Acertar T12 por coincidência (um SKU barato no Top-5 de popularidade) **não** significa que o V1 filtra orçamento. `e1_slice` ignora T11–T14 de propósito: um `overall` único faria o E1 parecer quebrado.

Campeão do mês plantado: **Clash** (`6D2W9K`) tem vendas altas em 01–24/08 e baixas na janela 25–31/08.


# 19. Possíveis evoluções arquiteturais

| Peça | Limitação do E1 que atacaria | Custo | Nesta disciplina |
| :--- | :--- | :--- | :--- |
| **Tool text-to-SQL** | Agregar 1 240 linhas no prompt; erro de janela 7d | 1 tool + allowlist | **E2 (primeiro incremento)** |
| **Tools preço / estoque** | T12; ruptura | 2 adapters; o agente passa a ver fatos mutáveis | E2, depois do SQL |
| **Workflow LangGraph** | Etapas fixas: interpretar → consultar fatos → montar Top-5 | estado + `max_steps` | obrigatório no E2 |
| **RAG / fichas** | T11, T13, T14 (claims) | ingestão + groundedness | quando o SQL não resolver necessidade |
| **ReAct** | ordem das tools não for fixa | mais tokens; modos de falha de tool | só se o workflow linear falhar |
| **MCP** | preço/estoque reusados por ≥2 nós | servidor extra | não no E2 se tool local bastar |
| **Memória** | restrições acumuladas entre turnos | custo de contexto | **descartada agora** (sessão 1-shot); só com caso que falhe sem estado |
| **Auditor de fairness / multiagente** | concentração residual além do piso RF-07 (n_brands ≥ 2) | supervisor + especialista | E3, se o piso de duas marcas não bastar |
| **Planejamento** | query composta (orçamento + claim + lançamento) | mais chamadas | se o workflow E2 errar compostos |

Não usar peça por moda. Empate ou piora no eval é resultado válido.


# 20. Pergunta obrigatória

> **Como o grupo pretende demonstrar, ao final do curso, que a arquitetura final apresenta vantagens em relação ao baseline?**

Reexecutando o **mesmo** golden-set (`golden_revision`), o **mesmo** schema e as **mesmas** funções de verify, no **mesmo modelo** (`model_version`) e na mesma sessão, com o baseline stuffing intacto.

Evidência de vantagem (hipótese):

- `e1_slice` (T01–T10) **não piora**;
- T12 passa a respeitar teto com preço **praticado** (tool), não `base_price`;
- T11/T13/T14 passam a acertar claim com citação da ficha (RAG), sem inventar;
- T05 continua a excluir o campeão do mês (SQL na janela 25–31/08, não stuffing);
- T01/T02/T04/T05 mantêm **n_brands ≥ 2** (RF-07); tools podem *facilitar* o mesmo requisito, não criá-lo do zero;
- latência, `llm_calls`, `tool_calls` e custo são **reportados** (não precisam ser menores).

Não promover SQL, RAG ou multiagente sem essa régua. Empate/piora numa dimensão entra no relatório com hipótese.


# 21. Conclusão

O RecFair ataca vitrine enviesada por popularidade e claims sem evidência. O **baseline V1 é parcial**: stuffing de catálogo + vendas diárias, Top-5 na janela **25–31/08/2026**, uma chamada Gemini (`model_version`), schema largo com preço/estoque/claim nulos.

A régua de 14 casos já contém o que o V1 deve acertar (categoria, abstenção, janela 7d, **diversidade mínima**) e o que deve falhar (preço, anticaspa, vegano, alta fixação). Copiar o Top-5 de popularidade em cabelos (5× Match) **reprova** T01. O próximo incremento justificado pelo tamanho da tabela é a **tool text-to-SQL** sobre as mesmas CSVs/`tb_*` no SQLite.


---

# Checklist antes da entrega

- [x] O problema está claramente definido.
- [x] O usuário-alvo foi identificado.
- [x] Escopo e não-objetivos estão explícitos.
- [x] Existem requisitos funcionais **verificáveis**.
- [x] Existem requisitos não funcionais.
- [x] Cada critério de sucesso diz **como** será medido.
- [x] O tipo de baseline foi classificado e justificado.
- [ ] O baseline executa sem erros *(Run all neste notebook)*.
- [x] Existem pelo menos três casos de teste, cobrindo mais de um tipo (14 casos).
- [x] Modelo, temperatura e data da execução estão registrados (`model_version`).
- [ ] Latência, tokens e número de chamadas foram registrados *(após Run all)*.
- [ ] Os resultados estão na tabela e interpretados *(após Run all)*.
- [x] As limitações foram analisadas.
- [x] A pergunta obrigatória foi respondida.
- [ ] O notebook foi executado do início ao fim e **salvo com as saídas**.
- [x] O notebook pode ser executado por outra pessoa (`generate_catalog.py` na mesma pasta).
- [x] Nenhuma chave de API foi incluída no notebook.
